# Voxel Encoding

In [1]:
%%capture
! pip install himalaya

In [20]:
!pip install --upgrade scipy

You should consider upgrading via the '/gpfs/home1/scur0412/NeuroNLP/neuro/bin/python -m pip install --upgrade pip' command.


In [1]:
import os
import numpy as np
from scipy.io import loadmat
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from scipy.stats import pearsonr
from himalaya.kernel_ridge import KernelRidgeCV
from himalaya.scoring import correlation_score 

In [2]:
mat_exp2 = "/home/scur0412/NeuroNLP/data/participants/M02/data_243sentences.mat"
mat_exp3 = "/home/scur0412/NeuroNLP/data/participants/M02/data_384sentences.mat"

embed_dir_exp2 = "/home/scur0412/NeuroNLP/results/embeddings/Qwen_Embedder/Experiment2_243"
embed_dir_exp3 = "/home/scur0412/NeuroNLP/results/embeddings/Qwen_Embedder/Experiment3_384"

llm_dir_exp2 = "/home/scur0412/NeuroNLP/results/embeddings/Qwen/Experiment2_243"
llm_dir_exp3 = "/home/scur0412/NeuroNLP/results/embeddings/Qwen/Experiment3_384"
num_layers = 37

In [3]:
def get_language_responses(mat_path):
    data = loadmat(mat_path, simplify_cells=True)
    target_index = np.where(data['meta']['atlases'] == 'languageLH')[0].item()
    column_indexes = np.concatenate([arr - 1 for arr in data['meta']['roiColumns'][target_index]], axis=0)
    # brain_responses = data['examples_passages'][:, column_indexes]
    brain_responses = data['examples_passagesentences'][:, column_indexes]
    print(f"Dimensionality of brain responses: {brain_responses.shape}")
    return brain_responses

In [4]:
y_exp2 = get_language_responses(mat_exp2)
y_exp3 = get_language_responses(mat_exp3)


/home/scur0412/NeuroNLP/neuro/lib64/python3.9/site-packages/scipy/io/matlab/_mio.py:227: MatReadWarning: Duplicate variable name "None" in stream - replacing previous with new
Consider mio5.varmats_from_mat to split file into single variable files
  matfile_dict = MR.get_variables(variable_names)


Dimensionality of brain responses: (243, 4930)
Dimensionality of brain responses: (384, 4930)


In [5]:
Y = np.concatenate([y_exp2, y_exp3], axis=0)
n_splits = 5
kf = KFold(n_splits=n_splits)
alphas = np.logspace(1, 20, 20)
_, n_voxels = Y.shape

layer_performances = {}

In [7]:
for layer_idx in range(num_layers):
    x_exp2 = np.load(os.path.join(embed_dir_exp2, f"qwen_layer{layer_idx}.npy"))
    x_exp3 = np.load(os.path.join(embed_dir_exp3, f"qwen_layer{layer_idx}.npy"))
    
    X = np.concatenate([x_exp2, x_exp3], axis=0)

    accs_train = np.empty((n_splits, n_voxels))
    accs_test = np.empty((n_splits, n_voxels))

    for i, (train_index, test_index) in enumerate(kf.split(X)):

        X_train, X_test = X[train_index, :], X[test_index, :]
        Y_train, Y_test = Y[train_index, :], Y[test_index, :]

        pipeline = make_pipeline(StandardScaler(with_mean=True, with_std=False),
                           KernelRidgeCV(alphas=alphas, cv=KFold(n_splits=5)))
        
        pipeline.fit(X_train, Y_train)

        preds_train = pipeline.predict(X_train)
        accs_train[i] = correlation_score(Y_train, preds_train)

        preds_test = pipeline.predict(X_test)
        accs_test[i] = correlation_score(Y_test, preds_test)

    mean_test_acc = accs_test.mean(axis=0).mean()
    mean_train_acc = accs_train.mean(axis=0).mean()
    layer_performances[layer_idx] = mean_test_acc
    
    print(f"Layer {layer_idx} Test Accuracy: {mean_test_acc:.4f}")
    print(f"Layer {layer_idx} Train Accuracy: {mean_train_acc:.4f}")


Layer 0 Test Accuracy: 0.0000
Layer 0 Train Accuracy: 0.0000
Layer 1 Test Accuracy: 0.1714
Layer 1 Train Accuracy: 0.3673
Layer 2 Test Accuracy: 0.1862
Layer 2 Train Accuracy: 0.4068
Layer 3 Test Accuracy: 0.2002
Layer 3 Train Accuracy: 0.4309
Layer 4 Test Accuracy: 0.2223
Layer 4 Train Accuracy: 0.4569
Layer 5 Test Accuracy: 0.2291
Layer 5 Train Accuracy: 0.4637
Layer 6 Test Accuracy: 0.2453
Layer 6 Train Accuracy: 0.4515
Layer 7 Test Accuracy: 0.2483
Layer 7 Train Accuracy: 0.4469
Layer 8 Test Accuracy: 0.2377
Layer 8 Train Accuracy: 0.4127
Layer 9 Test Accuracy: 0.2391
Layer 9 Train Accuracy: 0.4115
Layer 10 Test Accuracy: 0.2333
Layer 10 Train Accuracy: 0.4172
Layer 11 Test Accuracy: 0.2335
Layer 11 Train Accuracy: 0.4246
Layer 12 Test Accuracy: 0.2333
Layer 12 Train Accuracy: 0.4273
Layer 13 Test Accuracy: 0.2326
Layer 13 Train Accuracy: 0.4257
Layer 14 Test Accuracy: 0.2281
Layer 14 Train Accuracy: 0.4235
Layer 15 Test Accuracy: 0.2265
Layer 15 Train Accuracy: 0.4237
Layer 16 Tes

In [8]:
output_dir = "/home/scur0412/NeuroNLP/results"
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, "qwen_layer_performances.npy")
np.save(output_file, layer_performances)